# GEOG 592 — Module 7 Homework
## GeoPandas Practice



#### 1. Complete the hierarchy:
1. Shapely geometry
2. GeoSeries
3. GeoDataframe
#### 2. What type of object is returned by each expression? Write Shapely geometry, GeoSeries, GeoDataFrame, or pandas Series.
##### parks.geometry
GeoSeries
##### parks.geometry.iloc[0]
Shapely geometry
##### parks.iloc[0]
pandas Series
##### parks.iloc[[0]]
GeoDataFrame
#### 3. Explain in one sentence why `parks.geometry.iloc[0]` does not behave exactly like `parks.geometry`.
parks.geometry.iloc[0] is very specifically only one geometry whereas parks.geometry contains a series of geometries.
#### 4. Write code to display all column names in `parks` as a regular Python list.
```python
list(parks.columns)
```
#### 5. Write code to select only the `NAME` and `geometry` columns from `parks`.
```python
parks[["Name", "geometry"]]
```
#### 6. Write code to select the first three rows of `parks` while keeping the result as a GeoDataFrame.
```python
parks.head(3)
```
#### 7. Write code to inspect the geometry type of every feature in `parks`.
```python
parks.geom_type
```
#### 8. What does `parks.crs` tell you?
parks.crs will tell us which coordinate reference system (CRS) that the data is stored in because differences can change where the data is actually on Earth.
#### 9. Why can one point still produce a complete graph when you call `.plot()`?
.plot() creates a Matplotlib plot for the spatial feature, even if there is only one geometry.
#### 10. Assume `one_park = parks.geometry.iloc[0]`. What library provides the geometry object stored in `one_park`?
Shapely
```python
stops.crs
# EPSG:4326
```
#### 11. Would EPSG:4326 be a good CRS for measuring distance in feet? Explain briefly.
No because EPSG.4326 is measured in degrees so measurements in feet are not possible. Instead you would have to reproject the data with a different CRS that store units in feet.
#### 12. Write code to create `stops_nc` by reprojecting `stops` to EPSG:2264.
```python
stops_nc = stops.to_crs(epsg=2264)
```
#### 13. Write code to create an `area_sqft` column in `parks_nc` using polygon geometry area.
```python
parks_nc["area_sqrt"] = parks_nc.geometry.area
```
#### 14. Write code to calculate the distance from every stop in `stops_nc` to a single polygon named `park_1`.
```python
stops_nc.["distance_to_park1"] = stops_nc.geometry.distance(park_1)
```
#### 15. If `park_1` is a polygon, does `.distance(park_1)` measure distance to the polygon centroid? Explain.
No. The distance is calculated to the boundary/perimeter of the polygon, not to its centroid.
#### 16. Explain each
within: The entire left geometry is inside the right geometry\
contains: The left geometry contains the right geometry\
intersects: Two geometries have some shared point\
touches: The boundaries of two geometries touch but the interior does not overlap
#### 17. In `gpd.sjoin(parks, urban, predicate="within")`, which geometry is being tested as within which?
Testing which parks are within urban
#### 18. A park crosses an urban-area boundary. Would `within` be True? Would `intersects` be True?
Within would not be true becuase it has to hold the entirety of the park. However, intersects is true because intersects only requires for parks of the geometry to be within the other.
#### 19. Write code that returns how many stops are inside each polygon in a gpd named `parks` that has three polygons.
```python
parks.geometry.apply(stops.geometry.within).sum(axis=1)
```
```python
urban_parks = gpd.sjoin(parks, urban, predicate="within")
```
#### 20. What is the purpose of `gpd.sjoin()`?
gpd.sjoin() joins two GeoDataFrames based on their spatial relationship rather than a shared attribute field.
#### 21. Why is a spatial join different from a regular pandas join on a common ID field?
An attribute join matches records using a shared field, while a spatial join matches records using location.
#### 22. Write a spatial join that keeps bus stops that fall inside park polygons. Use `stops` as the left GeoDataFrame and `parks` as the right GeoDataFrame.
```python
stops_parks = gpd.sjoin(
    stops,
    parks,
    how="inner",
    predicate="within"
)
```
#### 23. After a point-in-polygon spatial join, how could you count how many bus stops were matched to each park? Describe the idea or write code.
```python
stops_in_parks["index_right"].value_counts()

stops_in_parks = gpd.sjoin(
    stops,
    parks,
    predicate="within"
)

stops_in_parks["index_right"].value_counts()
```
#### 24. Why can a polygon-to-polygon `intersects` spatial join create multiple output rows if each original layer was one polygon in size?
A spatial join creates one  row for each match. If one polygon intersects multiple polygons in the other layer, that polygon will appear in multiple output rows.
#### 25. You want to answer: “Which parks are within 500 feet of a bus stop?” Write how would you do this with GeoPandas
##### a. using a distance and apply method.
```python
within_500 = parks_nc.geometry.apply(
    lambda park: stops_nc.geometry.distance(park).min() <= 500
)

parks_nc[within_500]
```
##### b. using buffers
```python
stop_buffers = stops_nc.copy()
stop_buffers["geometry"] = stops_nc.buffer(500)

nearby_parks = gpd.sjoin(
    parks_nc,
    stop_buffers,
    predicate="intersects"
)
```